# QAOA for Max-Cut Optimisation

**Learning objectives**

- Encode a combinatorial objective
- Build a one-layer QAOA circuit
- Optimize parameters and sample solutions

## Environment setup



In [ ]:
# Run once in a fresh Colab/Jupyter environment, then restart the kernel if prompted.
%pip install -q qiskit qiskit-aer matplotlib pylatexenc scipy sympy

In [ ]:
from importlib.metadata import version
for package in ["qiskit", "qiskit-aer", "matplotlib", "pylatexenc", "scipy", "sympy"]:
    print(f"{package:15}: {version(package)}")

## Problem and circuit

We maximize the cut of a triangle graph. This educational implementation uses exact statevectors during optimization.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
edges=[(0,1),(1,2),(0,2)]; n=3
def cut_value(bits): return sum(bits[i]!=bits[j] for i,j in edges)
def circuit(gamma,beta,measure=False):
    qc=QuantumCircuit(n,n if measure else 0); qc.h(range(n))
    for i,j in edges: qc.cx(i,j); qc.rz(gamma,j); qc.cx(i,j)
    for q in range(n): qc.rx(2*beta,q)
    if measure: qc.measure(range(n),range(n))
    return qc
def expected(x):
    probs=Statevector.from_instruction(circuit(*x)).probabilities_dict()
    return sum(p*cut_value(bits) for bits,p in probs.items())
res=minimize(lambda x:-expected(x),x0=[0.8,0.5],method="COBYLA",options={"maxiter":100})
print(res.x,"expected cut",-res.fun); display(circuit(*res.x).draw("mpl"))

## Sample optimized circuit

The most likely strings represent candidate partitions; complementary strings describe the same cut.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
qc=circuit(*res.x,measure=True); counts=AerSimulator().run(qc,shots=4096).result().get_counts()
ranked=sorted(counts.items(),key=lambda x:x[1],reverse=True)
print([(b,c,cut_value(b)) for b,c in ranked]); display(plot_histogram(counts))